# 🔬 GNOT RF Cavity Neural Operator — Colab Runner

**Branch:** `feature/orthogonality-and-depth`  
**Yenilikler:**
- Per-mode frequency prediction (ayrı freq branch kaldırıldı)
- Orthogonality loss (modlar arası diklik kısıtı)
- Fizik-bilgili kayıp fonksiyonu (PINN boundary + peak-weighted MSE)

**Loss Fonksiyonu:**
$$\mathcal{L}_{total} = \mathcal{L}_{field} + \alpha \cdot \mathcal{L}_{freq} + \lambda \cdot \mathcal{L}_{bnd} + \beta \cdot \mathcal{L}_{ortho}$$

## 1. GPU Kontrolü & Ortam Kurulumu

In [ ]:
# GPU kontrol
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Repo klonla (veya Google Drive'dan bağla)
import os

REPO_URL = "https://github.com/KorayGokceler/rf_cavity_neural_operator.git"
BRANCH = "feature/orthogonality-and-depth"
REPO_DIR = "/content/rf_cavity_neural_operator"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull origin {BRANCH}

%cd {REPO_DIR}
print(f"\nÇalışma dizini: {os.getcwd()}")
!git log --oneline -3

In [ ]:
# Bağımlılıkları kur
!pip install -q scikit-fem[all] gmsh pytorch-lightning torchmetrics seaborn tqdm pyyaml h5py

## 2. Veri Hazırlığı

Eğer verin yoksa:
1. **Veri üret** (dataset_generator.py → .h5)
2. **Features çıkar** (dataset_converter.py → .pkl)

Eğer verin varsa **(Google Drive'dan)**: 3. adıma atla.

In [ ]:
# === OPSIYON A: Veri Üretimi (sıfırdan) ===
# Bu ~15-30 dakika sürer (1000 geometri)

GENERATE_DATA = False  # True yap eğer sıfırdan veri üreteceksen

if GENERATE_DATA:
    # 1. Mesh + FEM çözümleri üret
    !python -c "
from src.data_gen.dataset_generator import generate_dataset
from src.config import load_config
cfg = load_config('configs/default.yaml')
generate_dataset(cfg.data_gen)
"
    
    # 2. Features çıkar ve GNOT formatına dönüştür
    !python convert.py --config configs/default.yaml
    
    print("\n✅ Veri üretimi tamamlandı!")

In [ ]:
# === OPSIYON B: Google Drive'dan veri yükle ===

LOAD_FROM_DRIVE = True  # True yap eğer Drive'dan yükleyeceksen

if LOAD_FROM_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Drive'daki veri yolunu güncelle:
    DRIVE_PKL = "/content/drive/MyDrive/rf_cavity_data/gnot_dataset.pkl"  # ← BUNU GÜNCELLE
    
    os.makedirs("data", exist_ok=True)
    if os.path.exists(DRIVE_PKL):
        !cp "{DRIVE_PKL}" data/gnot_dataset.pkl
        print(f"✅ Veri kopyalandı: {DRIVE_PKL} → data/gnot_dataset.pkl")
    else:
        print(f"❌ Dosya bulunamadı: {DRIVE_PKL}")
        print("Drive'daki doğru yolu gir!")

In [ ]:
# Veri kontrolü
import pickle
import numpy as np

DATA_PATH = "data/gnot_dataset.pkl"

with open(DATA_PATH, "rb") as f:
    data = pickle.load(f)

n_geoms = len(data['geometry_pool'])
n_samples = len(data['samples'])

# Feature boyutu
first_geom = list(data['geometry_pool'].values())[0]
n_features = first_geom['Input_funcs'].shape[1]

# Mod dağılımı
modes = [int(s['Theta'][0]) for s in data['samples']]
mode_counts = {m: modes.count(m) for m in sorted(set(modes))}

print(f"📊 Dataset Özeti:")
print(f"   Geometri sayısı: {n_geoms}")
print(f"   Toplam sample:   {n_samples}")
print(f"   Feature boyutu:  {n_features}")
print(f"   Mod dağılımı:    {mode_counts}")
print(f"   Node aralığı:    {min(g['X'].shape[0] for g in data['geometry_pool'].values())} - {max(g['X'].shape[0] for g in data['geometry_pool'].values())}")

## 3. Eğitim Konfigürasyonu

In [ ]:
# Tüm ayarlar tek merkezden (Single Source of Truth): configs/default.yaml üzerinden okunur.
# Sadece Google Colab'in 2 CPU çekirdeğine takılmamak için eğitim komutunda num_workers=2 override edilecektir.
import yaml
with open('configs/default.yaml', 'r') as f:
    cfg = yaml.safe_load(f)
print(f"Model \t: n_shared={cfg['model']['n_shared_layers']}, n_mode={cfg['model']['n_mode_layers']}, predict_freq={cfg['model']['predict_frequency']}")
print(f"Kayıp \t: freq_w={cfg['training']['freq_weight']}, ortho_w={cfg['training'].get('ortho_weight', 0.0)}")

## 4. Hızlı Pipeline Testi

Tam eğitim başlamadan önce 1 iterasyon çalıştırarak her şeyin doğru olduğunu doğrula.

In [ ]:
# Fast dev run: 1 training + 1 validation iterasyonu
!python train.py --config configs/default.yaml --override training.num_workers=2 --fast_dev_run

print("\n✅ Pipeline testi başarılı! Tüm bileşenler çalışıyor.")

## 5. Tam Eğitim

In [ ]:
# Tam eğitim başlat
!python train.py --config configs/default.yaml --override training.num_workers=2

In [ ]:
# TensorBoard ile eğitim takibi (paralel çalıştır)
%load_ext tensorboard
%tensorboard --logdir training_logs

## 6. Inference & Görselleştirme

In [ ]:
# En iyi checkpoint'u bul
import glob

ckpt_pattern = "training_logs/gnot_old_data_v1/**/best-*.ckpt"
ckpts = glob.glob(ckpt_pattern, recursive=True)

if ckpts:
    best_ckpt = sorted(ckpts)[-1]
    print(f"En iyi checkpoint: {best_ckpt}")
else:
    print("❌ Checkpoint bulunamadı. Eğitim tamamlandı mı?")
    best_ckpt = None

In [ ]:
# Inference çalıştır
if best_ckpt:
    !python infer.py \
        --checkpoint "{best_ckpt}" \
        --data_path data/gnot_dataset.pkl \
        --split test \
        --num_samples 10 \
        --output_dir inference_results

In [ ]:
# Sonuçları göster
from IPython.display import Image, display
import glob

result_files = sorted(glob.glob("inference_results/*.png"))[:5]

for f in result_files:
    print(f"\n{'='*60}")
    print(f"📄 {os.path.basename(f)}")
    print(f"{'='*60}")
    display(Image(filename=f, width=800))

## 7. Ortogonalite Kontrolü

Modların gerçekten birbirine dik olup olmadığını kontrol edelim.

In [ ]:
import torch
import numpy as np
from src.data.dataset import GNOTDataset, gnot_collate_fn
from src.training.lightning_module import GNOTLightning
from torch.utils.data import DataLoader

if best_ckpt:
    # Model yükle
    model = GNOTLightning.load_from_checkpoint(best_ckpt)
    model.eval()
    if torch.cuda.is_available():
        model = model.cuda()

    # Test verisi
    dataset = GNOTDataset("data/gnot_dataset.pkl", split="test")
    loader = DataLoader(dataset, batch_size=64, shuffle=False, collate_fn=gnot_collate_fn)
    batch = next(iter(loader))
    if torch.cuda.is_available():
        batch = {k: v.cuda() if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

    with torch.no_grad():
        outputs = model(batch)

    pred = outputs['field']
    geom_ids = batch['geom_id'].squeeze(-1)
    mode_ids = batch['Theta_in'].squeeze(-1)
    mask = batch['Mask']
    node_area = batch['Input_funcs'][:, :, 5]

    # Her geometri için mod çiftlerinin cosine similarity'sini hesapla
    print("\n" + "="*60)
    print("ORTOGONALITE KONTROLÜ")
    print("="*60)
    
    all_cos = []
    for g_id in geom_ids.unique()[:10]:  # İlk 10 geometri
        g_mask = (geom_ids == g_id)
        g_idx = g_mask.nonzero(as_tuple=True)[0]
        if len(g_idx) < 2:
            continue
        
        g_modes = mode_ids[g_idx]
        g_preds = pred[g_idx]
        w = (node_area[g_idx[0]] * mask[g_idx[0]].float()).unsqueeze(-1)
        
        cos_vals = []
        for i in range(len(g_idx)):
            for j in range(i+1, len(g_idx)):
                if g_modes[i] == g_modes[j]:
                    continue
                u_i, u_j = g_preds[i], g_preds[j]
                dot = (u_i * u_j * w).sum()
                n_i = (u_i**2 * w).sum().sqrt().clamp(min=1e-8)
                n_j = (u_j**2 * w).sum().sqrt().clamp(min=1e-8)
                cos = (dot / (n_i * n_j)).abs().item()
                cos_vals.append(cos)
                all_cos.append(cos)
        
        modes_str = [str(m.item()) for m in g_modes]
        avg_cos = np.mean(cos_vals) if cos_vals else 0
        status = '✅' if avg_cos < 0.1 else ('⚠️' if avg_cos < 0.3 else '❌')
        print(f"  Geom {g_id.item():4d} | Modlar: {modes_str} | Avg |cos|: {avg_cos:.4f} {status}")
    
    print(f"\n{'─'*60}")
    print(f"  GENEL: Ortalama |cosine similarity|: {np.mean(all_cos):.4f}")
    print(f"  (0.0 = mükemmel diklik, 1.0 = paralel)")
    print(f"{'='*60}")
else:
    print("Checkpoint bulunamadı, önce eğitimi çalıştır.")

## 8. Checkpoint'u Google Drive'a Kaydet

In [ ]:
# Eğitilmiş modeli Drive'a kaydet
SAVE_TO_DRIVE = True

if SAVE_TO_DRIVE and best_ckpt:
    drive_save_dir = "/content/drive/MyDrive/rf_cavity_checkpoints"
    os.makedirs(drive_save_dir, exist_ok=True)
    
    save_name = os.path.basename(best_ckpt)
    !cp "{best_ckpt}" "{drive_save_dir}/{save_name}"
    
    # Inference sonuçlarını da kaydet
    !cp -r inference_results "{drive_save_dir}/inference_results"
    
    print(f"✅ Checkpoint kaydedildi: {drive_save_dir}/{save_name}")
    print(f"✅ Inference sonuçları kaydedildi: {drive_save_dir}/inference_results/")